In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import math

# --------------------------------
# 1. TRAFFIC PARAMETERS
# --------------------------------

N_ROADS = 4
SIMULATION_TIME = 3600

ARRIVAL_RATES = np.array([0.1, 0.15, 0.3, 0.2])
SATURATION_FLOW = 0.5

INITIAL_QUEUE = np.array([5, 10, 15, 8])

MIN_GREEN = 10
MAX_GREEN = 60
YELLOW_TIME = 3

# --------------------------------
# 2. PSO PARAMETERS
# --------------------------------

POPULATION = 50
ITERATIONS = 100

w = 0.7
c1 = 1.5
c2 = 1.5

rng = np.random.default_rng(42)

# Generate the same vehicle arrivals
# for every particle to ensure fair comparison.

arrivals = rng.poisson(
    ARRIVAL_RATES,
    size=(SIMULATION_TIME, N_ROADS)
)


# --------------------------------
# 3. TRAFFIC SIMULATION
# --------------------------------

def simulate(green_times, record=False):

    green_times = np.clip(
        np.rint(green_times), MIN_GREEN, MAX_GREEN
    ).astype(int)

    cycle_lengths = green_times + YELLOW_TIME
    cumulative = np.cumsum(cycle_lengths)
    cycle_time = cumulative[-1]

    queue = INITIAL_QUEUE.copy()
    service_credit = 0.0

    total_waiting_time = 0
    total_vehicles = INITIAL_QUEUE.sum() + arrivals.sum()

    if record:
        queue_history = np.zeros((SIMULATION_TIME, N_ROADS))

    for t in range(SIMULATION_TIME):

        # Add incoming vehicles
        queue += arrivals[t]

        # Identify the current signal phase
        cycle_position = t % cycle_time

        road = np.searchsorted(
            cumulative, cycle_position, side="right"
        )

        phase_start = (
            0 if road == 0 else cumulative[road - 1]
        )

        # Check whether the signal is green
        green_active = (
            cycle_position < phase_start + green_times[road]
        )

        if green_active:

            # Accumulate vehicle discharge capacity
            if queue[road] > 0:
                service_credit += SATURATION_FLOW

                departing = int(service_credit)
                served = min(queue[road], departing)

                queue[road] -= served
                service_credit -= served
            else:
                service_credit = 0.0

        # Calculate total vehicle waiting time
        total_waiting_time += queue.sum()

        if record:
            queue_history[t] = queue

    average_waiting_time = (
        total_waiting_time / total_vehicles
    )

    if record:
        return average_waiting_time, queue_history

    return average_waiting_time


# --------------------------------
# 4. FITNESS FUNCTION
# --------------------------------

def fitness(green_times):

    waiting_time = simulate(green_times)

    return 1 / (1 + waiting_time)


# --------------------------------
# 5. INITIALIZE PARTICLES
# --------------------------------

positions = rng.integers(
    MIN_GREEN,
    MAX_GREEN + 1,
    size=(POPULATION, N_ROADS)
).astype(float)

velocities = rng.uniform(
    -2, 2, size=(POPULATION, N_ROADS)
)

pbest = positions.copy()
pbest_fitness = np.array([
    fitness(p) for p in positions
])

best_index = np.argmax(pbest_fitness)

gbest = pbest[best_index].copy()
gbest_fitness = pbest_fitness[best_index]

convergence = [
    (1 / gbest_fitness) - 1
]


# --------------------------------
# 6. PSO OPTIMIZATION
# --------------------------------

for iteration in range(ITERATIONS):

    for i in range(POPULATION):

        r1 = rng.random(N_ROADS)
        r2 = rng.random(N_ROADS)

        # Update velocity
        velocities[i] = (
            w * velocities[i]
            + c1 * r1 * (pbest[i] - positions[i])
            + c2 * r2 * (gbest - positions[i])
        )

        # Update position
        positions[i] += velocities[i]

        # Apply green-time constraints
        positions[i] = np.clip(
            np.rint(positions[i]),
            MIN_GREEN,
            MAX_GREEN
        )

        # Evaluate new solution
        current_fitness = fitness(positions[i])

        # Update personal best
        if current_fitness > pbest_fitness[i]:

            pbest[i] = positions[i].copy()
            pbest_fitness[i] = current_fitness

        # Update global best
        if current_fitness > gbest_fitness:

            gbest = positions[i].copy()
            gbest_fitness = current_fitness

    best_wait = (1 / gbest_fitness) - 1
    convergence.append(best_wait)

    print(
        f"Iteration {iteration + 1}: "
        f"Best Average Waiting Time = {best_wait:.2f} seconds"
    )


# --------------------------------
# 7. FINAL RESULTS
# --------------------------------

optimal_wait, queue_history = simulate(
    gbest, record=True
)

print("\nOPTIMAL GREEN SIGNAL DURATIONS")

for i in range(N_ROADS):
    print(f"Road {i + 1}: {int(gbest[i])} seconds")

print(
    f"\nAverage Waiting Time: {optimal_wait:.2f} seconds"
)

print(
    f"Total Cycle Time: "
    f"{int(sum(gbest) + N_ROADS * YELLOW_TIME)} seconds"
)


# --------------------------------
# 8. GRAPH 1: PSO CONVERGENCE
# --------------------------------

plt.figure(figsize=(9, 5))

plt.plot(convergence)

plt.xlabel("Iteration")
plt.ylabel("Average Waiting Time (seconds)")
plt.title("PSO Convergence")

plt.grid(True)
plt.tight_layout()
plt.show()


# --------------------------------
# 9. GRAPH 2: OPTIMAL GREEN TIMES
# --------------------------------

plt.figure(figsize=(8, 5))

plt.bar(
    [f"Road {i+1}" for i in range(N_ROADS)],
    gbest
)

plt.xlabel("Roads")
plt.ylabel("Green Signal Duration (seconds)")
plt.title("Optimized Traffic Signal Timings")

plt.tight_layout()
plt.show()


# --------------------------------
# 10. GRAPH 3: QUEUE LENGTH
# --------------------------------

plt.figure(figsize=(10, 5))

for i in range(N_ROADS):

    plt.plot(
        queue_history[:, i],
        label=f"Road {i+1}"
    )

plt.xlabel("Simulation Time (seconds)")
plt.ylabel("Waiting Vehicles")
plt.title("Traffic Queue Length Over Time")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


Iteration 1: Best Average Waiting Time = 672.81 seconds
Iteration 2: Best Average Waiting Time = 671.19 seconds
Iteration 3: Best Average Waiting Time = 671.19 seconds
Iteration 4: Best Average Waiting Time = 670.49 seconds
Iteration 5: Best Average Waiting Time = 670.12 seconds
Iteration 6: Best Average Waiting Time = 670.12 seconds
Iteration 7: Best Average Waiting Time = 670.12 seconds
Iteration 8: Best Average Waiting Time = 669.80 seconds
Iteration 9: Best Average Waiting Time = 669.80 seconds
Iteration 10: Best Average Waiting Time = 669.80 seconds
Iteration 11: Best Average Waiting Time = 669.80 seconds
Iteration 12: Best Average Waiting Time = 669.80 seconds
Iteration 13: Best Average Waiting Time = 669.80 seconds
Iteration 14: Best Average Waiting Time = 669.80 seconds
Iteration 15: Best Average Waiting Time = 669.80 seconds
Iteration 16: Best Average Waiting Time = 669.80 seconds
Iteration 17: Best Average Waiting Time = 669.80 seconds
Iteration 18: Best Average Waiting Time 

KeyboardInterrupt: 